# HST / JWST MAST download notebook

Interactive helpers for querying MAST, inspecting sky coverage, and downloading
science products. Shared logic lives in `common.mast` and `nbutils`.


In [ ]:
import sys
from pathlib import Path

# Repo root on sys.path so `common` / `nbutils` import when run from notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import shapely
from astropy import units as u
from astropy.coordinates import SkyCoord
from astroquery.mast import Observations

from common.mast import (
    collect_hst_products,
    coverage_fraction,
    download_jwst_observations,
    filter_hst_observations,
    filter_jwst_observations,
    filter_jwst_products,
    polygons_from_obs_table,
    query_hst,
    query_jwst,
    query_region,
)
from nbutils import input_list, organize_reduction_tables, pick_deepest_images

## Target


In [ ]:
# Example targets (uncomment one)
# ra, dec = 259.2800254, 43.13566          # M92
# ra, dec = 24.174049, 15.78346            # NGC 628
# ra, dec = 210.803, 54.34906              # NGC 5457
ra, dec = 189.9976, -11.623               # NGC 4536

coord = SkyCoord(ra, dec, frame="icrs", unit=u.deg)
radius = 12 * u.arcmin
coord, radius

## HST query and download


In [ ]:
# Convenience wrapper (optional galaxy-size radius via use_galaxy_size=True)
hst_table = query_hst(
    coord,
    radius=radius,
    filters=["F275W", "F555W", "F814W"],  # None to keep all imaging filters
)
hst_table

In [ ]:
# Or query broadly then filter yourself
raw = query_region(coord, radius)
hst_table = filter_hst_observations(raw, filters=None)
print(len(hst_table), "HST imaging observations")
hst_table[:5]

In [ ]:
productlist = collect_hst_products(hst_table[:6])  # subset for a quick test
productlist

In [ ]:
# Uncomment to download
# Observations.download_products(productlist, extension="fits")

## JWST query, coverage, and download


In [ ]:
jwst_table = query_jwst(coord, radius=radius)
print(len(jwst_table), "JWST NIRCam imaging observations")
jwst_table[:5]

In [ ]:
pgons, filters = polygons_from_obs_table(jwst_table)
net_field = shapely.unary_union(pgons)
net_field

In [ ]:
want = ["F090W", "F115W", "F150W", "F200W"]
mask = [str(f) in want for f in filters]
print("coverage fraction:", coverage_fraction(pgons, mask))
shapely.unary_union(np.array(pgons, dtype=object)[mask])

In [ ]:
# Preview products for one observation
stage = 2  # 2=CAL, 3=I2D
plist = filter_jwst_products(Observations.get_product_list(jwst_table[0]), stage=stage)
plist

In [ ]:
# Uncomment to download all filtered JWST products
# download_jwst_observations(jwst_table, outdir=f"jwst_data/{'target'}", stage=stage)

## Local image bookkeeping (`nbutils`)


In [ ]:
# After downloading FITS files into the working directory:
# images = sorted(Path(".").glob("*_cal.fits"))
# obstable = input_list([str(p) for p in images])
# tables = organize_reduction_tables(obstable, byvisit=True, bymodule=True)
# deepest = pick_deepest_images([str(p) for p in images])
# obstable